##### Universidad Autónoma de Guadalajara
Aprendizaje Automatizado para la Gestión de Datos Masivos   

Profesor: Paulo López Meyer. 

Estudiante: Anayansi Cristina Hernández Abrego. 

Código vistos en clase ( tarea 11). 

In [2]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim.models.doc2vec import TaggedDocument
import string

#nltk.download('punkt')
#nltk.download('stopwords')

# Cargar el conjunto de datos
data = pd.read_csv(r'newsCorpora-trimmed.csv', encoding='utf-8') 
data.columns = ['category', 'text']

# --- AGREGA ESTAS LÍNEAS DE LIMPIEZA AQUÍ ---
# Elimina filas donde el texto o la categoría estén vacíos
data = data.dropna(subset=['text', 'category'])

# Asegúrate de que todos los valores en 'text' sean tratados como strings
data['text'] = data['text'].astype(str)
# -----------------------------------------

# Filtrar solo las categorías de interés
categories = ['b', 't', 'e', 'm']  # Business, Science and Technology, Entertainment, Health
data = data[data['category'].isin(categories)]

# Preprocesamiento de texto
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalpha() and word not in stop_words]
    return tokens

data['tokens'] = data['text'].apply(preprocess_text)

# --- NUEVA LÍNEA DE PROTECCIÓN ---
# Elimina filas que se hayan quedado sin palabras válidas tras el filtro
data = data[data['tokens'].apply(len) > 0]
#---------

# Crear documentos etiquetados
tagged_data = [TaggedDocument(words=row['tokens'], tags=[row['category']]) for index, row in data.iterrows()]


In [ ]:
from gensim.models import Doc2Vec

print(tagged_data)
#print("\n")

# Entrenar el modelo Doc2Vec
#model = Doc2Vec(tagged_data, vector_size=100, window=5, min_count=5, workers=4, epochs=20)

# MEJORADO
# vector_size=300: Vectores más grandes capturan mejor el significado (estándar en la industria).
# dm=1: Activa el modo "Distributed Memory" (PV-DM), que preserva el orden de las palabras.
#model = Doc2Vec(tagged_data, vector_size=300, dm=1, window=8, min_count=2, workers=4, epochs=20)
model = Doc2Vec(tagged_data, vector_size=300, dm=0, window=8, min_count=2, workers=4, epochs=50)


# Guardar el modelo
model.save("doc2vec_model")

In [ ]:
# Tercer chunk modificado
from sklearn.model_selection import train_test_split
# CAMBIO CLAVE: Usamos LinearSVC en lugar de SVC
from sklearn.svm import LinearSVC 
from sklearn.metrics import accuracy_score
import time

# 1. Vectorizar rápido usando comprensión de listas
print("1/3. Iniciando vectorización de documentos...")
inicio_vec = time.time()
X = [model.infer_vector(tokens) for tokens in data['tokens']]
y = data['category'].tolist()
print(f"✓ Vectorización completada en {time.time() - inicio_vec:.2f} segundos.")

# 2. Partición del dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Entrenar el clasificador ultra rápido (LinearSVC)
print("\n2/3. Entrenando clasificador LinearSVC (Optimizado para grandes volúmenes)...")
inicio_train = time.time()
# dual=False se recomienda cuando el número de muestras es mayor al número de características (dimensiones=100)
#classifier = LinearSVC(dual=False, random_state=42) 

# Optimizado 
classifier = LinearSVC(C=0.1, dual=False, random_state=42)
# Optimizado 2
#classifier = LinearSVC(class_weight='balanced', dual=False, random_state=42)

classifier.fit(X_train, y_train)
print(f"✓ Entrenamiento completada en {time.time() - inicio_train:.2f} segundos.")

# 4. Evaluar el clasificador
print("\n3/3. Evaluando el modelo...")
y_pred = classifier.predict(X_test)
print(f"Accuracy final: {accuracy_score(y_test, y_pred):.4f}")


1/3. Iniciando vectorización de documentos...


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


✓ Vectorización completada en 317.83 segundos.

2/3. Entrenando clasificador LinearSVC (Optimizado para grandes volúmenes)...
✓ Entrenamiento completada en 106.21 segundos.

3/3. Evaluando el modelo...
Accuracy final: 0.9159


In [ ]:
# CUARTO CHUCK MODIFICADO
def classify_new_document(text):
    # 1. Preprocesar el texto (convertir a tokens y quitar stopwords)
    tokens = preprocess_text(text)
    
    # 2. Convertir los tokens a vector directamente con el modelo entrenado
    vector = model.infer_vector(tokens)
    
    # 3. Realizar la predicción con el clasificador LinearSVC
    return classifier.predict([vector])[0]

# --- Ejemplo de clasificación ---
new_document = "New breakthrough in cancer research"
predicted_category = classify_new_document(new_document)

# Diccionario opcional para traducir las categorías
categorias_dict = {
    'b': 'Business (Negocios)',
    't': 'Science and Technology (Ciencia y Tecnología)',
    'e': 'Entertainment (Entretenimiento)',
    'm': 'Health (Salud)'
}

print(f"Documento: '{new_document}'")
print(f"Categoría predicha: {categorias_dict.get(predicted_category, predicted_category)}")


Documento: 'New breakthrough in cancer research'
Categoría predicha: Health (Salud)
